# Experiment 1 · Notebook 06: Sensitivity check, Partial Correct dropped

*J26-IT-304 PP1. Baseline confusion classifier (TF-IDF + Logistic Regression). Section 12.*

**Why this check exists.** The primary mapping treats `Partial Correct` like `Wrong` (both "confused"). An examiner will
ask: *"why treat partially correct as confused?"* This notebook re-runs the experiment with the Partial Correct rows
**removed** (Correct -> understood, Wrong -> confused) and reports both mappings side by side. If they are close, the mapping
choice is robust. If they diverge, that is itself a finding about where the label boundary sits: the EDA already found the
Partial / Wrong boundary noisy (notebook 01, section 3.6: 3 of 5 sampled Partial answers read as Wrong). **The primary
mapping stays the headline; this is a robustness note.** After this notebook the metrics are final.

**Same questions on each side.** The split is *not* redrawn. The primary split's question assignment is reused
(`apply_split`), so the test questions are identical and any difference comes from the label mapping alone.

Section numbers run continuously across the Experiment 1 series, so a reference such as "EDA §3.5" or "section 8" points to the same place in every notebook:

| Notebook | Sections | Stage |
|---|---|---|
| `01_data_and_eda` | §1-§3 | provenance, load, exploratory analysis, leakage demonstration |
| `02_cleaning_and_labels` | §4-§5 | deduplication, group key, label mapping, proxy caveat |
| `03_split_and_baselines` | §6-§7 | group-aware split, trivial baselines, frozen OOD probe |
| `04_model_training_and_evaluation` | §8-§9 | TF-IDF + LR, group-aware CV, the single test evaluation |
| `05_features_and_ood_probe` | §10-§11 | coefficient inspection, stability, length, OOD probe evaluation |
| `06_sensitivity_check` | §12 | robustness of the label mapping: Partial Correct dropped |

Each notebook runs on its own from a clean kernel (kernel `exp1-confusion`). Its first code cell rebuilds its inputs with the same `confusion_baseline` functions the previous stage used, so the stages cannot drift apart. Run them in order 01 → 06 to regenerate every file in `reports/experiments/`.

**Setup.** Imports and this stage's inputs: the primary split rebuilt and **proved identical** by its fingerprint, then the
`drop_partial` frame assigned to train and test **by the primary test questions**. The assertions show that no question is on both
sides and that the new test questions are a subset of the primary ones (a test question whose only answers were Partial
would disappear, never move).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from confusion_baseline import REPORTS_DIR, SEED
from confusion_baseline.data import load_raw
from confusion_baseline.evaluate import metrics
from confusion_baseline.model import build_pipeline, cv_folds, cv_search, length_baseline, load_params, save_params, token_count
from confusion_baseline.prep import clean, map_labels
from confusion_baseline.split import apply_split, group_split, split_fingerprint

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)
plt.rcParams.update({"font.size": 11})
FIG_DIR = REPORTS_DIR / "experiments"

# ---- primary split, as in notebooks 03-05 ----
df, followup_quarantine, _ = load_raw()
del followup_quarantine
clean_df, _ = clean(df)
y_primary = map_labels(clean_df, "primary")
train, test = group_split(y_primary, test_size=0.2, seed=SEED)
U4_FINGERPRINT = "511fadfecbff1c84dad9f22c4ecdd51052b13c6c1187c6f60d2468342905e44c"
fp = split_fingerprint(test)
assert fp == U4_FINGERPRINT, fp
print(f"PASS: same primary test set as notebooks 03-05 (fingerprint {fp[:12]}...)")

# ---- drop_partial frame, SAME question assignment ----
dp = map_labels(clean_df, "drop_partial")
print(f"drop_partial frame: {len(dp)} rows (Partial Correct rows removed: {dp.attrs['dropped_partial']}); "
      f"{(dp['y'] == 0).sum()} understood, {(dp['y'] == 1).sum()} confused")
assert len(dp) == 787
train_dp, test_dp = apply_split(dp, set(test["group_key"]))

shared = set(train_dp["group_key"]) & set(test_dp["group_key"])
assert not shared, shared
assert set(test_dp["group_key"]) <= set(test["group_key"])
assert set(train_dp["group_key"]) <= set(train["group_key"])
assert len(train_dp) + len(test_dp) == len(dp)
print(f"PASS: train_dp and test_dp share 0 group_keys; test_dp groups are a subset of the primary test groups "
      f"({test_dp['group_key'].nunique()} of {test['group_key'].nunique()}); train_dp groups a subset of the primary train groups")

sizes = pd.DataFrame([
    {"mapping": "primary", "split": "train", "rows": len(train), "groups": train["group_key"].nunique(), "confused share": round(float(train["y"].mean()), 4)},
    {"mapping": "primary", "split": "test", "rows": len(test), "groups": test["group_key"].nunique(), "confused share": round(float(test["y"].mean()), 4)},
    {"mapping": "drop_partial", "split": "train", "rows": len(train_dp), "groups": train_dp["group_key"].nunique(), "confused share": round(float(train_dp["y"].mean()), 4)},
    {"mapping": "drop_partial", "split": "test", "rows": len(test_dp), "groups": test_dp["group_key"].nunique(), "confused share": round(float(test_dp["y"].mean()), 4)},
])
display(sizes)

PASS: same primary test set as notebooks 03-05 (fingerprint 511fadfecbff...)
drop_partial frame: 787 rows (Partial Correct rows removed: 289); 420 understood, 367 confused
PASS: train_dp and test_dp share 0 group_keys; test_dp groups are a subset of the primary test groups (72 of 72); train_dp groups a subset of the primary train groups


,mapping,split,rows,groups,confused share
0,primary,train,860,288,0.6128
1,primary,test,216,72,0.5972
2,drop_partial,train,631,288,0.4723
3,drop_partial,test,156,72,0.4423


## 12. Sensitivity check: Partial Correct dropped

### 12.1 Fit everything on train_dp (the test set is not used yet)

**Why.** All fitting happens first, on `train_dp` only, so the test set is touched in exactly one cell later. The trivial
baselines are fitted here: the majority class (note that **understood may now be the majority**, in which case "most frequent"
predicts understood and scores F1(confused) = 0, and that is reported as it is), and the length-only logistic regression.
**Row (a)** is the *like-for-like* model: the same TF-IDF + LR configuration chosen in notebook 04 (`chosen_params_primary.json`),
fitted on `train_dp`, so only the labels differ. **Row (b)** re-tunes: the same `PARAM_GRID` is searched on `train_dp` with
group-aware folds, selected on macro F1.

In [2]:
from sklearn.dummy import DummyClassifier

X0_tr = np.zeros((len(train_dp), 1))                          # Dummy ignores features
maj = DummyClassifier(strategy="most_frequent").fit(X0_tr, train_dp["y"])
maj_class = int(maj.predict(X0_tr[:1])[0])
print(f"majority class in train_dp: {'confused' if maj_class == 1 else 'understood'} "
      f"(train_dp confused share {train_dp['y'].mean():.4f})")

len_dp = length_baseline(train_dp)

params_primary = load_params(FIG_DIR / "chosen_params_primary.json")
model_a = build_pipeline(**params_primary).fit(train_dp["Answer"], train_dp["y"])
print("row (a) 'primary config' params:", params_primary)

majority class in train_dp: understood (train_dp confused share 0.4723)
row (a) 'primary config' params: {'clf__C': 100, 'clf__class_weight': 'balanced', 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 2), 'tfidf__stop_words': None, 'tfidf__sublinear_tf': True}


**Why.** Re-tuning (row b) must obey the same rules as the primary run: group-aware folds inside `train_dp`. The 5 PASS lines prove
that each fold's validation questions are disjoint from its training questions.

In [3]:
folds_dp = cv_folds(train_dp)
for k, (tr_idx, va_idx) in enumerate(folds_dp, 1):
    overlap = set(train_dp["group_key"].iloc[tr_idx]) & set(train_dp["group_key"].iloc[va_idx])
    assert not overlap, overlap
    print(f"PASS: fold {k}: {train_dp['group_key'].iloc[tr_idx].nunique()} train groups / "
          f"{train_dp['group_key'].iloc[va_idx].nunique()} validation groups share 0 questions ({len(tr_idx)} / {len(va_idx)} rows)")

search_b = cv_search(train_dp)
res = pd.DataFrame(search_b.cv_results_)
res.drop(columns=[c for c in res.columns if "time" in c]).to_csv(FIG_DIR / "cv_results_drop_partial.csv", index=False)
print("saved: reports/experiments/cv_results_drop_partial.csv (timing columns dropped)")

short = {"param_tfidf__ngram_range": "ngram", "param_tfidf__min_df": "min_df", "param_tfidf__stop_words": "stop",
         "param_tfidf__sublinear_tf": "sublinear", "param_clf__C": "C", "param_clf__class_weight": "class_weight",
         "mean_test_macro_f1": "cv macro F1", "std_test_macro_f1": "sd", "mean_test_f1_confused": "cv F1(conf)"}
ranked = res.sort_values("mean_test_macro_f1", ascending=False, kind="stable").reset_index(drop=True)
top10 = ranked.loc[:9, list(short)].rename(columns=short).round(4)
top10[["stop", "class_weight"]] = top10[["stop", "class_weight"]].astype(object).fillna("None")   # None is stored as NaN
top10.index = range(1, 11)
print("\nTop 10 configs by mean CV macro F1 on train_dp:")
display(top10)
best_mean, best_sd = ranked.loc[0, "mean_test_macro_f1"], ranked.loc[0, "std_test_macro_f1"]
print(f"configs within one sd ({best_sd:.4f}) of the best mean ({best_mean:.4f}): "
      f"{int((ranked['mean_test_macro_f1'] >= best_mean - best_sd).sum())} of {len(ranked)}")

print("\nCHOSEN (re-tuned) hyperparameters:")
for k, v in search_b.best_params_.items():
    print(f"  {k:22s} {v!r}")
print(f"mean CV macro F1 = {search_b.best_score_:.4f}")
same = {k: (v if not isinstance(v, tuple) else v) for k, v in search_b.best_params_.items()} == params_primary
print("re-tuned params identical to the primary config:", same)
save_params(search_b.best_params_, FIG_DIR / "chosen_params_drop_partial.json", selection_metric="macro_f1",
            cv=f"StratifiedGroupKFold(n_splits=5, shuffle=True, random_state={SEED}), groups=group_key, on train_dp",
            mean_cv_macro_f1=round(float(search_b.best_score_), 6))
print("saved: reports/experiments/chosen_params_drop_partial.json")
model_b = search_b.best_estimator_

PASS: fold 1: 230 train groups / 58 validation groups share 0 questions (505 / 126 rows)
PASS: fold 2: 228 train groups / 60 validation groups share 0 questions (503 / 128 rows)
PASS: fold 3: 231 train groups / 57 validation groups share 0 questions (506 / 125 rows)
PASS: fold 4: 235 train groups / 53 validation groups share 0 questions (504 / 127 rows)
PASS: fold 5: 228 train groups / 60 validation groups share 0 questions (506 / 125 rows)


saved: reports/experiments/cv_results_drop_partial.csv (timing columns dropped)

Top 10 configs by mean CV macro F1 on train_dp:


,ngram,min_df,stop,sublinear,C,class_weight,cv macro F1,sd,cv F1(conf)
1,"(1, 1)",1,None,True,10.0,None,0.6453,0.0496,0.6115
2,"(1, 1)",2,None,False,10.0,balanced,0.6447,0.0330,0.6237
3,"(1, 1)",2,None,False,1.0,None,0.6445,0.0584,0.5863
4,"(1, 1)",1,None,False,10.0,None,0.6426,0.0483,0.6096
5,"(1, 2)",2,None,False,100.0,None,0.6423,0.0587,0.5994
6,"(1, 1)",1,None,False,10.0,balanced,0.6422,0.0486,0.6164
7,"(1, 1)",1,None,True,100.0,balanced,0.6420,0.0318,0.6156
8,"(1, 1)",1,None,True,10.0,balanced,0.6415,0.0495,0.6138
9,"(1, 2)",1,None,True,1.0,None,0.6411,0.0588,0.5544
10,"(1, 2)",2,None,False,100.0,balanced,0.6407,0.0594,0.6048


configs within one sd (0.0496) of the best mean (0.6453): 118 of 160

CHOSEN (re-tuned) hyperparameters:
  clf__C                 10
  clf__class_weight      None
  tfidf__min_df          1
  tfidf__ngram_range     (1, 1)
  tfidf__stop_words      None
  tfidf__sublinear_tf    True
mean CV macro F1 = 0.6453
re-tuned params identical to the primary config: False
saved: reports/experiments/chosen_params_drop_partial.json


### 12.2 The test set, once

**Why. The test set is touched here, in this one cell.** It scores every system on `test_dp` (the Correct and Wrong answers of the
same 72 primary test questions, or fewer if a question had only Partial answers), prints the report and confusion matrix for
row (a), writes `comparison_drop_partial.csv`, and runs the cluster bootstrap over test **questions** (1000 resamples, seed 42).
The **gate**: if either TF-IDF row reaches F1(confused) > 0.90, stop and investigate before concluding, since on this data
that is more likely leakage than success. "Always confused" is shown explicitly because it is the F1(confused) floor.

In [4]:
from sklearn.metrics import classification_report, confusion_matrix

y_dp = test_dp["y"].to_numpy()
X0_te = np.zeros((len(test_dp), 1))
preds = {
    "Majority class (most frequent)": maj.predict(X0_te),
    "Always confused": np.ones(len(test_dp), dtype=int),
    "Stratified random (seed 42)": DummyClassifier(strategy="stratified", random_state=SEED).fit(X0_tr, train_dp["y"]).predict(X0_te),
    "Length-only LR": len_dp.predict(token_count(test_dp["Answer"])),
    "TF-IDF + LR (a) primary config": model_a.predict(test_dp["Answer"]),
    "TF-IDF + LR (b) re-tuned": model_b.predict(test_dp["Answer"]),
}
cols = {"accuracy": "Accuracy", "precision_confused": "Precision", "recall_confused": "Recall",
        "f1_confused": "F1 (confused)", "macro_f1": "Macro F1"}
m_all = {n: metrics(y_dp, p) for n, p in preds.items()}
comparison = pd.DataFrame([{"System": n, **{cols[k]: round(v, 4) for k, v in m.items()}} for n, m in m_all.items()])

for name in ["TF-IDF + LR (a) primary config", "TF-IDF + LR (b) re-tuned"]:
    f1 = m_all[name]["f1_confused"]
    print(f"GATE (> 0.90): {name}: F1(confused) = {f1:.4f} -> {'FIRED: stop and report' if f1 > 0.90 else 'not fired'}")

print(f"\nTEST (drop_partial): {len(test_dp)} rows, {test_dp['group_key'].nunique()} questions, confused share {y_dp.mean():.4f}")
print("\nCOMPARISON TABLE (test_dp):")
display(comparison)
comparison.to_csv(FIG_DIR / "comparison_drop_partial.csv", index=False)
print("saved: reports/experiments/comparison_drop_partial.csv")

print("\nclassification_report, row (a), threshold 0.5:")
print(classification_report(y_dp, preds["TF-IDF + LR (a) primary config"], target_names=["understood", "confused"], digits=4, zero_division=0))
cm = confusion_matrix(y_dp, preds["TF-IDF + LR (a) primary config"], labels=[0, 1])
print("confusion matrix, row (a), raw counts (rows = actual, cols = predicted):")
display(pd.DataFrame(cm, index=["actual understood", "actual confused"], columns=["pred understood", "pred confused"]))
tn, fp_, fn, tp = (int(x) for x in cm.ravel())
print(f"TP = {tp} | FP = {fp_} | FN = {fn} | TN = {tn}")

s_f1 = []
for s in range(1000):
    c = DummyClassifier(strategy="stratified", random_state=s).fit(X0_tr, train_dp["y"])
    s_f1.append(metrics(y_dp, c.predict(X0_te))["f1_confused"])
s_f1 = np.array(s_f1)
print(f"\nstratified baseline F1(confused) over seeds 0-999: mean {s_f1.mean():.4f} +/- {s_f1.std(ddof=1):.4f} (sd); the table row is seed {SEED}")

# ---- cluster bootstrap over test_dp GROUPS ----
rng = np.random.default_rng(SEED)
g_idx = {g: np.flatnonzero((test_dp["group_key"] == g).to_numpy()) for g in test_dp["group_key"].unique()}
garr = list(g_idx)
sel = {"a": preds["TF-IDF + LR (a) primary config"], "b": preds["TF-IDF + LR (b) re-tuned"],
       "len": preds["Length-only LR"], "const": preds["Always confused"]}
boot = {k: [] for k in sel}
for _ in range(1000):
    pick = rng.integers(0, len(garr), len(garr))
    idx = np.concatenate([g_idx[garr[i]] for i in pick])
    for k, pr in sel.items():
        boot[k].append(metrics(y_dp[idx], pr[idx]))

def col(k, metric):
    return np.array([b[metric] for b in boot[k]])

def report(label, arr, point):
    lo, hi = np.percentile(arr, [2.5, 97.5])
    verdict = "" if label.startswith("(a) ") and " - " not in label else ("  EXCLUDES 0" if (lo > 0 or hi < 0) else "  INCLUDES 0")
    print(f"  {label:48s} point {point:+.4f}   95% CI [{lo:+.4f}, {hi:+.4f}]{verdict}")

ma, mb, ml, mc = (m_all["TF-IDF + LR (a) primary config"], m_all["TF-IDF + LR (b) re-tuned"], m_all["Length-only LR"], m_all["Always confused"])
print(f"\nCLUSTER BOOTSTRAP over {len(garr)} test questions, 1000 resamples, seed {SEED}; 95% percentile intervals:")
report("(a) macro F1", col("a", "macro_f1"), ma["macro_f1"])
report("(a) F1 (confused)", col("a", "f1_confused"), ma["f1_confused"])
report("(a) - length-only, macro F1", col("a", "macro_f1") - col("len", "macro_f1"), ma["macro_f1"] - ml["macro_f1"])
report("(a) - always confused, F1 (confused)", col("a", "f1_confused") - col("const", "f1_confused"), ma["f1_confused"] - mc["f1_confused"])
report("(a) - length-only, F1 (confused)", col("a", "f1_confused") - col("len", "f1_confused"), ma["f1_confused"] - ml["f1_confused"])
report("(b) - (a), macro F1  [re-tuned vs primary config]", col("b", "macro_f1") - col("a", "macro_f1"), mb["macro_f1"] - ma["macro_f1"])
report("(b) - (a), F1 (confused)", col("b", "f1_confused") - col("a", "f1_confused"), mb["f1_confused"] - ma["f1_confused"])
ci_a = np.percentile(col("a", "macro_f1"), [2.5, 97.5])
print(f"\nwidth of row (a)'s macro-F1 95% interval: {ci_a[1] - ci_a[0]:.4f}; |(b) - (a)| macro F1 = {abs(mb['macro_f1'] - ma['macro_f1']):.4f}")

GATE (> 0.90): TF-IDF + LR (a) primary config: F1(confused) = 0.6131 -> not fired
GATE (> 0.90): TF-IDF + LR (b) re-tuned: F1(confused) = 0.6197 -> not fired

TEST (drop_partial): 156 rows, 72 questions, confused share 0.4423

COMPARISON TABLE (test_dp):


,System,Accuracy,Precision,Recall,F1 (confused),Macro F1
0,Majority class (most frequent),0.5577,0.0000,0.0000,0.0000,0.3580
1,Always confused,0.4423,0.4423,1.0000,0.6133,0.3067
2,Stratified random (seed 42),0.4808,0.4062,0.3768,0.3910,0.4692
3,Length-only LR,0.7244,0.6625,0.7681,0.7114,0.7238
4,TF-IDF + LR (a) primary config,0.6603,0.6176,0.6087,0.6131,0.6551
5,TF-IDF + LR (b) re-tuned,0.6538,0.6027,0.6377,0.6197,0.6510


saved: reports/experiments/comparison_drop_partial.csv

classification_report, row (a), threshold 0.5:
              precision    recall  f1-score   support

  understood     0.6932    0.7011    0.6971        87
    confused     0.6176    0.6087    0.6131        69

    accuracy                         0.6603       156
   macro avg     0.6554    0.6549    0.6551       156
weighted avg     0.6598    0.6603    0.6600       156

confusion matrix, row (a), raw counts (rows = actual, cols = predicted):


,pred understood,pred confused
actual understood,61,26
actual confused,27,42


TP = 42 | FP = 26 | FN = 27 | TN = 61



stratified baseline F1(confused) over seeds 0-999: mean 0.4574 +/- 0.0492 (sd); the table row is seed 42



CLUSTER BOOTSTRAP over 72 test questions, 1000 resamples, seed 42; 95% percentile intervals:
  (a) macro F1                                     point +0.6551   95% CI [+0.5685, +0.7369]
  (a) F1 (confused)                                point +0.6131   95% CI [+0.5079, +0.7087]
  (a) - length-only, macro F1                      point -0.0687   95% CI [-0.1721, +0.0481]  INCLUDES 0
  (a) - always confused, F1 (confused)             point -0.0002   95% CI [-0.1072, +0.1018]  INCLUDES 0
  (a) - length-only, F1 (confused)                 point -0.0983   95% CI [-0.2082, +0.0189]  INCLUDES 0
  (b) - (a), macro F1  [re-tuned vs primary config] point -0.0041   95% CI [-0.0771, +0.0856]  INCLUDES 0
  (b) - (a), F1 (confused)                         point +0.0066   95% CI [-0.0753, +0.1026]  INCLUDES 0

width of row (a)'s macro-F1 95% interval: 0.1685; |(b) - (a)| macro F1 = 0.0041


### 12.3 Side by side

**Why.** One table puts the primary mapping next to both `drop_partial` rows. The primary numbers are **read from
`comparison_primary.csv`** (notebook 04's saved table), not recomputed. F1(confused) is not comparable across mappings
because its floor (the "always confused" F1) moves with the confused base rate, so the table carries each mapping's own floor and
the margins over it.

In [5]:
saved = pd.read_csv(FIG_DIR / "comparison_primary.csv").set_index("System")
p_model, p_len, p_maj = saved.loc["TF-IDF + LR"], saved.loc["Length-only LR"], saved.loc["Majority class"]   # majority = always confused in primary

def side_row(mapping, n_tr, n_te, share, floor, len_macro, acc, f1, macro):
    return {"mapping": mapping, "train rows": n_tr, "test rows": n_te, "test confused share": round(share, 4),
            "always-confused F1(confused)": round(floor, 4), "length-only macro F1": round(len_macro, 4),
            "TF-IDF+LR accuracy": round(acc, 4), "F1(confused)": round(f1, 4), "macro F1": round(macro, 4),
            "margin over always-confused (F1 confused)": round(f1 - floor, 4),
            "margin over length-only (macro F1)": round(macro - len_macro, 4)}

side = pd.DataFrame([
    side_row("primary", len(train), len(test), float(test["y"].mean()), float(p_maj["F1 (confused)"]), float(p_len["Macro F1"]),
             float(p_model["Accuracy"]), float(p_model["F1 (confused)"]), float(p_model["Macro F1"])),
    side_row("drop_partial (a) primary config", len(train_dp), len(test_dp), float(y_dp.mean()), mc["f1_confused"], ml["macro_f1"],
             ma["accuracy"], ma["f1_confused"], ma["macro_f1"]),
    side_row("drop_partial (b) re-tuned", len(train_dp), len(test_dp), float(y_dp.mean()), mc["f1_confused"], ml["macro_f1"],
             mb["accuracy"], mb["f1_confused"], mb["macro_f1"]),
])
side.to_csv(FIG_DIR / "sensitivity_side_by_side.csv", index=False)
print("SIDE-BY-SIDE (primary values read from comparison_primary.csv):")
display(side.set_index("mapping").T)
print("saved: reports/experiments/sensitivity_side_by_side.csv")

SIDE-BY-SIDE (primary values read from comparison_primary.csv):


mapping,primary,drop_partial (a) primary config,drop_partial (b) re-tuned
train rows,860.0000,631.0000,631.0000
test rows,216.0000,156.0000,156.0000
test confused share,0.5972,0.4423,0.4423
always-confused F1(confused),0.7478,0.6133,0.6133
length-only macro F1,0.6214,0.7238,0.7238
TF-IDF+LR accuracy,0.6898,0.6603,0.6538
F1(confused),0.7581,0.6131,0.6197
macro F1,0.6629,0.6551,0.6510
margin over always-confused (F1 confused),0.0103,-0.0002,0.0064
margin over length-only (macro F1),0.0415,-0.0687,-0.0728


saved: reports/experiments/sensitivity_side_by_side.csv


### 12.4 Reading the result

**F1(confused) is not comparable across the two mappings.** The confused share drops from about 60% (test 0.597) to about 44% (test
0.442), so the floor moves with it: "always confused" scores F1(confused) 0.748 under the primary mapping and 0.613 under
`drop_partial`. The honest comparison is macro F1 and the margins over each mapping's own floors.

**1. The TF-IDF model is robust to the mapping, and Correct vs Wrong is not easier for it.** Macro F1 is **0.663** under the
primary mapping (95% CI 0.595 to 0.731, notebook 04) and **0.655** with Partial Correct removed (95% CI 0.569 to 0.737). The
difference is -0.008, far smaller than the intervals are wide (about 0.14 and 0.17), so the two are indistinguishable. Removing the
ambiguous middle class does **not** make the task clearly easier for the lexical model. Its margin over its own F1(confused) floor stays at
about zero in both mappings (+0.010 primary, **-0.0002** here, CI -0.107 to +0.102), so on F1(confused) the model never clearly beats
"always confused". The answer to "why treat partial as confused?" is therefore that it **does not change what the model can do**.

**2. It changes what the length baseline can do, and that is the finding about the boundary.** Length-only LR improves from macro F1
**0.621 to 0.724** when Partial Correct is removed. Correct and Wrong answers differ in length more than Partial does (EDA 3.4:
median 22 / 19 / 16 tokens), so Partial sits between them and blurs the length signal. As a result the margin of TF-IDF + LR over
length-only **flips from +0.042 (primary) to -0.069** (CI -0.172 to +0.048, includes 0). Neither sign is distinguishable from 0 with 156
rows, but the direction says that on the cleanest contrast (Correct vs Wrong) **a word count is at least as good as the vocabulary
model**. This is consistent with notebook 05: the lexical signal is weak and partly generator style.

**3. Where is the hard part of the primary task?** It is **not** mainly at the Partial boundary, at least for the lexical model: it
scores about the same with or without Partial, so even Correct vs Wrong is hard for it (macro F1 0.655, accuracy 0.660). What the Partial
boundary does is *obscure the length signal* and (EDA 3.6) hold some answers that read as Wrong. For PP2's labelling this suggests that
Partial Correct behaves more like a genuinely intermediate class (it sits between the others in length) than like random noise, although the EDA reading showed some Partial answers that read as Wrong, and that a model which captures **content correctness** is
needed beyond vocabulary: neither the vocabulary model nor a word count does that here.

**4. Re-tuning does not matter.** Row (b) re-selects a different configuration (unigrams, `C=10`, no class weighting; CV macro F1 0.645, with
118 of 160 configs within one sd, so the choice is weakly determined) and scores macro F1 0.651 against 0.655 for row (a). The difference
(-0.004, CI -0.077 to +0.086) is about 40 times smaller than the width of (a)'s interval (0.169). The primary configuration is as good a
choice here as a re-tuned one.

**The primary mapping stays the headline.** This is a robustness note: it shows the primary result is not an artefact of where Partial
Correct was assigned, and it shows that the length-only baseline is the stronger of the two simple systems on the cleaner contrast.

## Findings of notebook 06

| Topic | Finding | Where |
|---|---|---|
| Setup | Same 72 test questions as the primary run (`apply_split`, no new split): `train_dp` 631 rows, `test_dp` 156 rows, 0 shared questions, test groups a subset of the primary test groups. Confused share 0.472 train / 0.442 test (vs 0.613 / 0.597). | Setup |
| Floors | "Always confused" F1(confused) is 0.613 here vs 0.748 in the primary mapping, and understood is now the majority class (majority predictor: F1(confused) = 0, macro F1 0.358). F1(confused) is **not comparable** across mappings. | §12.2 |
| **Robustness** | TF-IDF + LR macro F1 **0.655** (CI 0.569-0.737) vs **0.663** primary (CI 0.595-0.731): indistinguishable. Removing Partial Correct does **not** make the task easier for the lexical model; the F1(confused) margin over "always confused" is -0.0002 (CI -0.107 to +0.102). | §12.2, §12.3 |
| Length baseline | Length-only LR rises from macro F1 0.621 (primary) to **0.724** without Partial Correct, so Partial blurs the length signal. TF-IDF + LR minus length-only flips from +0.042 to **-0.069** (CI -0.172 to +0.048, includes 0). | §12.2, §12.3 |
| Partial boundary | The hard part for the lexical model is not mainly the Partial boundary: even Correct vs Wrong gives macro F1 0.655. Partial Correct behaves like an intermediate class (though some answers read as Wrong, 01 section 3.6); PP2 needs a content-aware signal. | §12.4 |
| Re-tuning | Re-tuned (b) chooses unigrams, `C=10`, no class weighting (CV macro F1 0.645; 118 of 160 configs within one sd) and scores 0.651 vs 0.655 for (a): difference -0.004, CI -0.077 to +0.086. Re-tuning does not matter. | §12.2 |
| Gate | Did not fire (F1(confused) 0.613 / 0.620, both far below 0.90). | §12.2 |
| Headline | The primary mapping stays the headline; the metrics are final after this notebook. | §12.4 |